# Baseline justo: ResNet puro vs DeMemte + VQ-VAE

Comparación justa usando exactamente la misma metodología para ambos modelos:
- Flowers102
- Stratified K-Fold
- Early stopping
- Mismo esquema de evaluación clean/noisy/masking

In [1]:
import os
import copy
import json
import random
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, Subset

import torchvision
import torchvision.transforms as transforms
from sklearn.model_selection import StratifiedKFold

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('cuda:', torch.cuda.is_available())

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
cuda: True


In [2]:
@dataclass
class Config:
    data_dir: str = '../data'
    num_classes: int = 102
    batch_size: int = 64
    num_workers: int = 2

    n_splits: int = 3
    cv_seed: int = 42

    lr_baseline: float = 1e-3
    lr_vq: float = 3e-4
    lr_cls: float = 1e-3
    weight_decay: float = 1e-4

    epochs_baseline_max: int = 30
    epochs_phase1_max: int = 10
    epochs_phase2_max: int = 30

    early_stop_patience: int = 3
    early_stop_min_delta: float = 1e-4

    scheduler_factor: float = 0.5
    scheduler_patience: int = 1

    feature_noise_std: float = 0.35

    embedding_dim: int = 128
    num_embeddings: int = 512
    commitment_cost: float = 0.25
    ema_decay: float = 0.99
    eps: float = 1e-5

    init_tau: float = 0.0
    init_alpha: float = 1.5
    init_memory_scale: float = 1.0

    denoise_weight: float = 1.0
    vq_weight: float = 1.0

    max_train_batches_debug: int = 20
    max_val_batches_debug: int = 10

    artifacts_root: str = './out/artifacts'
    experiment_name: str = 'baseline_vs_dememte_cv'

    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
device = torch.device(cfg.device)
criterion = nn.CrossEntropyLoss()
print(json.dumps(asdict(cfg), indent=2))

{
  "data_dir": "../data",
  "num_classes": 102,
  "batch_size": 64,
  "num_workers": 2,
  "n_splits": 3,
  "cv_seed": 42,
  "lr_baseline": 0.001,
  "lr_vq": 0.0003,
  "lr_cls": 0.001,
  "weight_decay": 0.0001,
  "epochs_baseline_max": 30,
  "epochs_phase1_max": 10,
  "epochs_phase2_max": 30,
  "early_stop_patience": 3,
  "early_stop_min_delta": 0.0001,
  "scheduler_factor": 0.5,
  "scheduler_patience": 1,
  "feature_noise_std": 0.35,
  "embedding_dim": 128,
  "num_embeddings": 512,
  "commitment_cost": 0.25,
  "ema_decay": 0.99,
  "eps": 1e-05,
  "init_tau": 0.0,
  "init_alpha": 1.5,
  "init_memory_scale": 1.0,
  "denoise_weight": 1.0,
  "vq_weight": 1.0,
  "max_train_batches_debug": 20,
  "max_val_batches_debug": 10,
  "artifacts_root": "./out/artifacts",
  "experiment_name": "baseline_vs_dememte_cv",
  "device": "cuda"
}


In [3]:
def _extract_labels(dataset):
    if hasattr(dataset, '_labels'):
        return np.array(dataset._labels)
    if hasattr(dataset, 'labels'):
        return np.array(dataset.labels)
    ys = []
    for i in range(len(dataset)):
        _, y = dataset[i]
        ys.append(int(y))
    return np.array(ys)

def build_datasets(config: Config):
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    transform_eval = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    tr = torchvision.datasets.Flowers102(root=config.data_dir, split='train', download=False, transform=transform_train)
    va = torchvision.datasets.Flowers102(root=config.data_dir, split='val', download=False, transform=transform_eval)
    te = torchvision.datasets.Flowers102(root=config.data_dir, split='test', download=False, transform=transform_eval)

    cv_ds = ConcatDataset([tr, va])
    cv_y = np.concatenate([_extract_labels(tr), _extract_labels(va)], axis=0)
    testloader = DataLoader(te, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=True)
    return cv_ds, cv_y, testloader

cv_dataset, cv_labels, testloader = build_datasets(cfg)
print('cv size:', len(cv_dataset), '| classes:', len(np.unique(cv_labels)))

cv size: 2040 | classes: 102


In [4]:
def make_backbone():
    base = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    return nn.Sequential(*list(base.children())[:-2])

class ResNetPureBaseline(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feat_norm = nn.LayerNorm(512)
        self.classifier = nn.Linear(512, num_classes)

    def _extract_backbone(self, x):
        with torch.no_grad():
            return self.backbone(x)

    def classify_from_features(self, feats):
        return self.classifier(self.feat_norm(self.pool(feats).flatten(1)))

    def forward(self, x, feature_noise_std=0.0, return_debug=False):
        clean_feats = self._extract_backbone(x)
        noisy_feats = clean_feats + feature_noise_std * torch.randn_like(clean_feats) if feature_noise_std > 0 else clean_feats
        logits = self.classify_from_features(noisy_feats)
        if return_debug:
            return logits, {'clean_feats': clean_feats, 'noisy_feats': noisy_feats}
        return logits

class VectorQuantizerEMA(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25, decay=0.99, eps=1e-5):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.decay = decay
        self.eps = eps
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1 / num_embeddings, 1 / num_embeddings)
        self.register_buffer('ema_cluster_size', torch.zeros(num_embeddings))
        self.register_buffer('ema_w', self.embedding.weight.data.clone())

    def forward(self, z_e):
        b, c, h, w = z_e.shape
        z_e_perm = z_e.permute(0, 2, 3, 1).contiguous()
        flat = z_e_perm.view(-1, c)
        emb = self.embedding.weight

        distances = flat.pow(2).sum(1, keepdim=True) - 2 * flat @ emb.t() + emb.pow(2).sum(1, keepdim=True).t()
        idx = torch.argmin(distances, dim=1)
        one_hot = F.one_hot(idx, self.num_embeddings).type(flat.dtype)
        q_flat = one_hot @ emb
        q = q_flat.view(b, h, w, c).permute(0, 3, 1, 2).contiguous()

        if self.training:
            n = one_hot.sum(0)
            dw = one_hot.t() @ flat
            self.ema_cluster_size.mul_(self.decay).add_(n, alpha=1 - self.decay)
            self.ema_w.mul_(self.decay).add_(dw, alpha=1 - self.decay)
            total = self.ema_cluster_size.sum()
            cluster_size = ((self.ema_cluster_size + self.eps) / (total + self.num_embeddings * self.eps)) * total
            self.embedding.weight.data.copy_(self.ema_w / cluster_size.unsqueeze(1))

        vq_loss = self.commitment_cost * F.mse_loss(z_e, q.detach())
        q_st = z_e + (q - z_e).detach()
        dq_map = ((z_e - q.detach()) ** 2).mean(dim=1, keepdim=True)
        return q_st, vq_loss, dq_map

class SpatialVQVAE(nn.Module):
    def __init__(self, in_channels=512, hidden_channels=256, embedding_dim=128, num_embeddings=512, commitment_cost=0.25, decay=0.99, eps=1e-5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, embedding_dim, 1),
        )
        self.vq = VectorQuantizerEMA(num_embeddings, embedding_dim, commitment_cost, decay, eps)
        self.decoder = nn.Sequential(
            nn.Conv2d(embedding_dim, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, in_channels, 1),
        )

    def forward(self, x):
        z = self.encoder(x)
        zq, vq_loss, dq_map = self.vq(z)
        x_rec = self.decoder(zq)
        return x_rec, vq_loss, dq_map

class DeMemteSpatialVQ(nn.Module):
    def __init__(self, backbone, vq_vae, num_classes):
        super().__init__()
        self.backbone = backbone
        self.vq_vae = vq_vae
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feat_norm = nn.LayerNorm(512)
        self.classifier = nn.Linear(512, num_classes)
        self.gate_tau = nn.Parameter(torch.tensor(0.0))
        self.gate_alpha = nn.Parameter(torch.tensor(1.5))
        self.memory_scale = nn.Parameter(torch.tensor(1.0))

    def _extract_backbone(self, x):
        with torch.no_grad():
            return self.backbone(x)

    @staticmethod
    def _normalize_per_sample(dq_map, eps=1e-6):
        m = dq_map.mean(dim=(1,2,3), keepdim=True)
        s = dq_map.std(dim=(1,2,3), keepdim=True)
        return (dq_map - m) / (s + eps)

    def classify_from_features(self, feats):
        rec_feats, _, dq_map = self.vq_vae(feats)
        dq_norm = self._normalize_per_sample(dq_map)
        alpha = F.softplus(self.gate_alpha)
        signal = torch.sigmoid(alpha * (self.gate_tau - dq_norm))
        enhanced = feats + self.memory_scale * signal * rec_feats
        out = self.classifier(self.feat_norm(self.pool(enhanced).flatten(1)))
        return out

    def forward(self, x, feature_noise_std=0.0, return_debug=False):
        clean_feats = self._extract_backbone(x)
        noisy_feats = clean_feats + feature_noise_std * torch.randn_like(clean_feats) if feature_noise_std > 0 else clean_feats
        rec_feats, vq_loss, dq_map = self.vq_vae(noisy_feats)
        denoise_loss = F.mse_loss(rec_feats, clean_feats)
        dq_norm = self._normalize_per_sample(dq_map)
        alpha = F.softplus(self.gate_alpha)
        signal = torch.sigmoid(alpha * (self.gate_tau - dq_norm))
        enhanced = noisy_feats + self.memory_scale * signal * rec_feats
        logits = self.classifier(self.feat_norm(self.pool(enhanced).flatten(1)))
        if return_debug:
            return logits, denoise_loss, vq_loss, {'clean_feats': clean_feats}
        return logits, denoise_loss, vq_loss

In [5]:
def make_baseline(config):
    bb = make_backbone().to(device)
    for p in bb.parameters():
        p.requires_grad = False
    bb.eval()
    m = ResNetPureBaseline(bb, config.num_classes).to(device)
    opt = optim.AdamW(m.classifier.parameters(), lr=config.lr_baseline, weight_decay=config.weight_decay)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=config.scheduler_factor, patience=config.scheduler_patience)
    return m, opt, sch

def make_dememte(config):
    bb = make_backbone().to(device)
    for p in bb.parameters():
        p.requires_grad = False
    bb.eval()
    vq = SpatialVQVAE(
        in_channels=512,
        hidden_channels=256,
        embedding_dim=config.embedding_dim,
        num_embeddings=config.num_embeddings,
        commitment_cost=config.commitment_cost,
        decay=config.ema_decay,
        eps=config.eps,
    ).to(device)
    m = DeMemteSpatialVQ(bb, vq, config.num_classes).to(device)
    m.gate_tau.data.fill_(config.init_tau)
    m.gate_alpha.data.fill_(config.init_alpha)
    m.memory_scale.data.fill_(config.init_memory_scale)

    p1 = list(m.vq_vae.parameters()) + [m.gate_tau, m.gate_alpha, m.memory_scale]
    opt1 = optim.AdamW(p1, lr=config.lr_vq, weight_decay=config.weight_decay)
    sch1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, mode='min', factor=config.scheduler_factor, patience=config.scheduler_patience)

    opt2 = optim.AdamW([
        {'params': m.vq_vae.parameters(), 'lr': config.lr_vq},
        {'params': m.classifier.parameters(), 'lr': config.lr_cls},
        {'params': [m.gate_tau, m.gate_alpha, m.memory_scale], 'lr': config.lr_vq},
    ], weight_decay=config.weight_decay)
    sch2 = optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='min', factor=config.scheduler_factor, patience=config.scheduler_patience)
    return m, opt1, sch1, opt2, sch2

def run_epoch_baseline(model, loader, optimizer, train):
    model.train(train)
    tot = {'loss': 0.0, 'acc': 0.0, 'n': 0}
    for bi, (x, y) in enumerate(loader):
        if train and bi >= cfg.max_train_batches_debug:
            break
        if (not train) and bi >= cfg.max_val_batches_debug:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        logits = model(x, feature_noise_std=cfg.feature_noise_std)
        loss = criterion(logits, y)
        if train:
            loss.backward()
            optimizer.step()
        bs = x.size(0)
        acc = (logits.argmax(1) == y).float().mean().item()
        tot['loss'] += loss.item() * bs
        tot['acc'] += acc * bs
        tot['n'] += bs
    tot['loss'] /= max(1, tot['n'])
    tot['acc'] /= max(1, tot['n'])
    return tot

def run_epoch_dememte_p1(model, loader, optimizer, train):
    model.train(train)
    tot = {'loss': 0.0, 'n': 0}
    for bi, (x, _) in enumerate(loader):
        if train and bi >= cfg.max_train_batches_debug:
            break
        if (not train) and bi >= cfg.max_val_batches_debug:
            break
        x = x.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        _, denoise_loss, vq_loss = model(x, feature_noise_std=cfg.feature_noise_std)
        loss = cfg.denoise_weight * denoise_loss + cfg.vq_weight * vq_loss
        if train:
            loss.backward()
            optimizer.step()
        bs = x.size(0)
        tot['loss'] += loss.item() * bs
        tot['n'] += bs
    tot['loss'] /= max(1, tot['n'])
    return tot

def run_epoch_dememte_p2(model, loader, optimizer, train):
    model.train(train)
    tot = {'loss': 0.0, 'acc': 0.0, 'n': 0}
    for bi, (x, y) in enumerate(loader):
        if train and bi >= cfg.max_train_batches_debug:
            break
        if (not train) and bi >= cfg.max_val_batches_debug:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        logits, denoise_loss, vq_loss = model(x, feature_noise_std=cfg.feature_noise_std)
        ce = criterion(logits, y)
        loss = ce + cfg.denoise_weight * denoise_loss + cfg.vq_weight * vq_loss
        if train:
            loss.backward()
            optimizer.step()
        bs = x.size(0)
        acc = (logits.argmax(1) == y).float().mean().item()
        tot['loss'] += loss.item() * bs
        tot['acc'] += acc * bs
        tot['n'] += bs
    tot['loss'] /= max(1, tot['n'])
    tot['acc'] /= max(1, tot['n'])
    return tot

def eval_clean_acc(model, loader, max_batches=10):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for bi, (x, y) in enumerate(loader):
            if bi >= max_batches:
                break
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if isinstance(model, ResNetPureBaseline):
                logits = model(x, feature_noise_std=0.0)
            else:
                logits, _, _ = model(x, feature_noise_std=0.0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / max(1, total)

In [ ]:
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
artifacts_dir = os.path.join(cfg.artifacts_root, f'{cfg.experiment_name}_{run_stamp}')
os.makedirs(artifacts_dir, exist_ok=True)

skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.cv_seed)
results = {'baseline': [], 'dememte': []}
best_baseline = {'val_acc': -1.0, 'fold': None, 'state': None}
best_dememte = {'val_acc': -1.0, 'fold': None, 'state': None}

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(cv_labels)), cv_labels), start=1):
    print(f'\n===== Fold {fold}/{cfg.n_splits} =====')
    tr_ds = Subset(cv_dataset, train_idx.tolist())
    va_ds = Subset(cv_dataset, val_idx.tolist())

    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

    # Baseline
    b_model, b_opt, b_sch = make_baseline(cfg)
    best_b_acc, best_b_state, no_imp = -1.0, copy.deepcopy(b_model.state_dict()), 0

    for ep in range(1, cfg.epochs_baseline_max + 1):
        tr = run_epoch_baseline(b_model, tr_loader, b_opt, train=True)
        va = run_epoch_baseline(b_model, va_loader, b_opt, train=False)
        b_sch.step(va['loss'])
        print(f"[Fold {fold} | Baseline {ep:02d}] tr_acc={tr['acc']:.4f} val_acc={va['acc']:.4f}")
        if va['acc'] > best_b_acc + cfg.early_stop_min_delta:
            best_b_acc = va['acc']
            best_b_state = copy.deepcopy(b_model.state_dict())
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= cfg.early_stop_patience:
                print(f'Baseline early stop en época {ep}.')
                break

    b_model.load_state_dict(best_b_state)
    b_val = eval_clean_acc(b_model, va_loader, max_batches=cfg.max_val_batches_debug)
    b_test = eval_clean_acc(b_model, testloader, max_batches=cfg.max_val_batches_debug)
    results['baseline'].append({'fold': fold, 'val_acc': float(b_val), 'test_acc_debug': float(b_test)})
    if b_val > best_baseline['val_acc']:
        best_baseline = {'val_acc': b_val, 'fold': fold, 'state': copy.deepcopy(best_b_state)}

    # DeMemte
    d_model, d_opt1, d_sch1, d_opt2, d_sch2 = make_dememte(cfg)

    best_p1, best_p1_state, no_imp1 = float('inf'), copy.deepcopy(d_model.state_dict()), 0
    for ep in range(1, cfg.epochs_phase1_max + 1):
        tr = run_epoch_dememte_p1(d_model, tr_loader, d_opt1, train=True)
        va = run_epoch_dememte_p1(d_model, va_loader, d_opt1, train=False)
        d_sch1.step(va['loss'])
        print(f"[Fold {fold} | DeMemte P1 {ep:02d}] tr_loss={tr['loss']:.4f} val_loss={va['loss']:.4f}")
        if va['loss'] < best_p1 - cfg.early_stop_min_delta:
            best_p1 = va['loss']
            best_p1_state = copy.deepcopy(d_model.state_dict())
            no_imp1 = 0
        else:
            no_imp1 += 1
            if no_imp1 >= cfg.early_stop_patience:
                print(f'DeMemte P1 early stop en época {ep}.')
                break

    d_model.load_state_dict(best_p1_state)

    best_p2, best_p2_state, no_imp2 = -1.0, copy.deepcopy(d_model.state_dict()), 0
    for ep in range(1, cfg.epochs_phase2_max + 1):
        tr = run_epoch_dememte_p2(d_model, tr_loader, d_opt2, train=True)
        va = run_epoch_dememte_p2(d_model, va_loader, d_opt2, train=False)
        d_sch2.step(va['loss'])
        print(f"[Fold {fold} | DeMemte P2 {ep:02d}] tr_acc={tr['acc']:.4f} val_acc={va['acc']:.4f}")
        if va['acc'] > best_p2 + cfg.early_stop_min_delta:
            best_p2 = va['acc']
            best_p2_state = copy.deepcopy(d_model.state_dict())
            no_imp2 = 0
        else:
            no_imp2 += 1
            if no_imp2 >= cfg.early_stop_patience:
                print(f'DeMemte P2 early stop en época {ep}.')
                break

    d_model.load_state_dict(best_p2_state)
    d_val = eval_clean_acc(d_model, va_loader, max_batches=cfg.max_val_batches_debug)
    d_test = eval_clean_acc(d_model, testloader, max_batches=cfg.max_val_batches_debug)
    results['dememte'].append({'fold': fold, 'val_acc': float(d_val), 'test_acc_debug': float(d_test)})
    if d_val > best_dememte['val_acc']:
        best_dememte = {'val_acc': d_val, 'fold': fold, 'state': copy.deepcopy(best_p2_state)}

    print(f'[Fold {fold}] baseline_val={b_val:.4f} | dememte_val={d_val:.4f}')

summary = {
    'config': asdict(cfg),
    'results': results,
    'best_baseline_fold': best_baseline['fold'],
    'best_baseline_val_acc': float(best_baseline['val_acc']),
    'best_dememte_fold': best_dememte['fold'],
    'best_dememte_val_acc': float(best_dememte['val_acc']),
}

summary_path = os.path.join(artifacts_dir, 'baseline_vs_dememte_cv_metrics.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

torch.save({'state_dict': best_baseline['state'], 'fold': best_baseline['fold'], 'config': asdict(cfg)}, os.path.join(artifacts_dir, 'best_baseline.pt'))
torch.save({'state_dict': best_dememte['state'], 'fold': best_dememte['fold'], 'config': asdict(cfg)}, os.path.join(artifacts_dir, 'best_dememte.pt'))

print('\n===== RESUMEN CV =====')
print('Baseline best fold/acc:', best_baseline['fold'], float(best_baseline['val_acc']))
print('DeMemte best fold/acc:', best_dememte['fold'], float(best_dememte['val_acc']))
print('artifact dir:', artifacts_dir)
print('summary:', summary_path)


===== Fold 1/3 =====
[Fold 1 | Baseline 01] tr_acc=0.1430 val_acc=0.4078
[Fold 1 | Baseline 02] tr_acc=0.6789 val_acc=0.6906
[Fold 1 | Baseline 03] tr_acc=0.8516 val_acc=0.7766
[Fold 1 | Baseline 04] tr_acc=0.9289 val_acc=0.8172
[Fold 1 | Baseline 05] tr_acc=0.9633 val_acc=0.8297
[Fold 1 | Baseline 06] tr_acc=0.9750 val_acc=0.8484
[Fold 1 | Baseline 07] tr_acc=0.9812 val_acc=0.8422
[Fold 1 | Baseline 08] tr_acc=0.9906 val_acc=0.8609
[Fold 1 | Baseline 09] tr_acc=0.9930 val_acc=0.8469
[Fold 1 | Baseline 10] tr_acc=0.9961 val_acc=0.8453
[Fold 1 | Baseline 11] tr_acc=0.9992 val_acc=0.8594
Baseline early stop en época 11.
[Fold 1 | DeMemte P1 01] tr_loss=2.2862 val_loss=1.8362
[Fold 1 | DeMemte P1 02] tr_loss=1.7976 val_loss=1.7858
[Fold 1 | DeMemte P1 03] tr_loss=1.7082 val_loss=1.7202
[Fold 1 | DeMemte P1 04] tr_loss=1.6803 val_loss=1.7052
[Fold 1 | DeMemte P1 05] tr_loss=1.6622 val_loss=1.6922
[Fold 1 | DeMemte P1 06] tr_loss=1.6513 val_loss=1.6691
[Fold 1 | DeMemte P1 07] tr_loss=1.64

In [ ]:
baseline_model, _, _ = make_baseline(cfg)
baseline_model.load_state_dict(best_baseline['state'])
dememte_model, _, _, _, _ = make_dememte(cfg)
dememte_model.load_state_dict(best_dememte['state'])

def eval_clean_noisy_mc(model, loader, noise_std, repeats=5, max_batches=10):
    model.eval()
    clean_scores, noisy_scores = [], []
    for rep in range(repeats):
        torch.manual_seed(1200 + rep)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(1200 + rep)
        clean_correct = 0
        noisy_correct = 0
        total = 0
        with torch.no_grad():
            for bi, (x, y) in enumerate(loader):
                if bi >= max_batches:
                    break
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                if isinstance(model, ResNetPureBaseline):
                    logits_c = model(x, feature_noise_std=0.0)
                    logits_n = model(x, feature_noise_std=noise_std)
                else:
                    logits_c, _, _ = model(x, feature_noise_std=0.0)
                    logits_n, _, _ = model(x, feature_noise_std=noise_std)
                clean_correct += (logits_c.argmax(1) == y).sum().item()
                noisy_correct += (logits_n.argmax(1) == y).sum().item()
                total += y.size(0)
        clean_scores.append(clean_correct / max(1, total))
        noisy_scores.append(noisy_correct / max(1, total))
    return {
        'clean_mean': float(np.mean(clean_scores)),
        'clean_std': float(np.std(clean_scores)),
        'noisy_mean': float(np.mean(noisy_scores)),
        'noisy_std': float(np.std(noisy_scores)),
    }

def eval_mask_corruption(model, loader, mask_ratio, repeats=5, max_batches=10):
    model.eval()
    clean_scores, masked_scores, changed_scores = [], [], []
    for rep in range(repeats):
        torch.manual_seed(2200 + rep)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(2200 + rep)
        clean_correct = 0
        masked_correct = 0
        changed = 0
        total = 0
        with torch.no_grad():
            for bi, (x, y) in enumerate(loader):
                if bi >= max_batches:
                    break
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                if isinstance(model, ResNetPureBaseline):
                    logits_clean, dbg = model(x, feature_noise_std=0.0, return_debug=True)
                    feats = dbg['clean_feats']
                    b, c, _, _ = feats.shape
                    keep = (torch.rand(b, c, 1, 1, device=feats.device) > mask_ratio).float()
                    logits_masked = model.classify_from_features(feats * keep)
                else:
                    logits_clean, _, _, dbg = model(x, feature_noise_std=0.0, return_debug=True)
                    feats = dbg['clean_feats']
                    b, c, _, _ = feats.shape
                    keep = (torch.rand(b, c, 1, 1, device=feats.device) > mask_ratio).float()
                    logits_masked = model.classify_from_features(feats * keep)

                pred_c = logits_clean.argmax(1)
                pred_m = logits_masked.argmax(1)
                clean_correct += (pred_c == y).sum().item()
                masked_correct += (pred_m == y).sum().item()
                changed += (pred_c != pred_m).sum().item()
                total += y.size(0)

        clean_scores.append(clean_correct / max(1, total))
        masked_scores.append(masked_correct / max(1, total))
        changed_scores.append(changed / max(1, total))

    return {
        'clean_mean': float(np.mean(clean_scores)),
        'clean_std': float(np.std(clean_scores)),
        'masked_mean': float(np.mean(masked_scores)),
        'masked_std': float(np.std(masked_scores)),
        'pred_change_rate_mean': float(np.mean(changed_scores)),
        'pred_change_rate_std': float(np.std(changed_scores)),
    }

noise_levels = [0.35, 0.5, 0.75, 1.0, 1.5]
mask_levels = [0.25, 0.5, 0.7]

baseline_noise = [dict(noise_std=n, **eval_clean_noisy_mc(baseline_model, testloader, n)) for n in noise_levels]
dememte_noise = [dict(noise_std=n, **eval_clean_noisy_mc(dememte_model, testloader, n)) for n in noise_levels]
baseline_mask = [dict(mask_ratio=m, **eval_mask_corruption(baseline_model, testloader, m)) for m in mask_levels]
dememte_mask = [dict(mask_ratio=m, **eval_mask_corruption(dememte_model, testloader, m)) for m in mask_levels]

b_vals = np.array([r['val_acc'] for r in results['baseline']], dtype=float)
d_vals = np.array([r['val_acc'] for r in results['dememte']], dtype=float)
folds = [r['fold'] for r in results['baseline']]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

ix = np.arange(len(folds))
w = 0.38
axes[0].bar(ix - w/2, b_vals, width=w, label='Baseline')
axes[0].bar(ix + w/2, d_vals, width=w, label='DeMemte')
axes[0].set_xticks(ix)
axes[0].set_xticklabels([f'fold {f}' for f in folds])
axes[0].set_ylim(0.0, 1.0)
axes[0].set_title('CV val acc')
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend()

xg = np.array(noise_levels, dtype=float)
axes[1].plot(xg, [x['noisy_mean'] for x in baseline_noise], marker='o', label='Baseline')
axes[1].plot(xg, [x['noisy_mean'] for x in dememte_noise], marker='s', label='DeMemte')
axes[1].set_title('Noisy accuracy vs noise_std')
axes[1].set_xlabel('noise_std')
axes[1].set_ylabel('accuracy')
axes[1].grid(alpha=0.3)
axes[1].legend()

xm = np.array(mask_levels, dtype=float)
axes[2].plot(xm, [x['masked_mean'] for x in baseline_mask], marker='o', label='Baseline')
axes[2].plot(xm, [x['masked_mean'] for x in dememte_mask], marker='s', label='DeMemte')
axes[2].set_title('Masked accuracy vs mask_ratio')
axes[2].set_xlabel('mask_ratio')
axes[2].set_ylabel('accuracy')
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print('Baseline CV mean±std:', float(np.mean(b_vals)), '±', float(np.std(b_vals)))
print('DeMemte CV mean±std:', float(np.mean(d_vals)), '±', float(np.std(d_vals)))